# Economic Backtest with Trading Costs

## Purpose

The statistical experiments evaluate forecast error and directional accuracy. This notebook asks a different question:

> **Do the predicted next-day returns have economic value after realistic trading costs?**

The notebook uses the saved **held-out ensemble predictions** from the completed phases. It does not re-train the forecasting models and does not tune a trading threshold on the test period.

## Two pre-specified trading rules

### 1. Daily sign strategy (primary sanity check)

- predicted return > 0: long (+1)
- predicted return < 0: short (-1)
- predicted return = 0: cash (0)

This is the simplest economic translation of directional forecasts. Trading cost is charged whenever the position changes. A long-to-short reversal therefore pays two one-way costs (exit + new entry).

### 2. Cost-aware strategy (secondary check)

For each day, the strategy chooses among `-1`, `0`, and `+1` by maximizing:

`predicted utility = position × predicted simple return - one-way cost × turnover`

This creates a transparent no-trade region automatically and avoids tuning a threshold using test outcomes.

## Cost sensitivity

Both policies are reported for one-way transaction costs of **0, 5, 10, 20, and 50 basis points**. This is preferable to claiming one arbitrary cost assumption is universally correct.

The notebook reports cumulative return, annualized return, annualized volatility, Sharpe ratio, maximum drawdown, turnover, and number of position changes. Buy-and-hold and cash are included as economic benchmarks.

> A negative backtest result is still useful: it would show that small statistical MAE improvements do not necessarily translate into tradable economic value.

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

PROJECT_DIR = Path("/content/drive/MyDrive/Crypto_Research")
OUTPUT_ROOT = PROJECT_DIR / "revised_outputs_v3"
BACKTEST_DIR = OUTPUT_ROOT / "economic_backtest"
BACKTEST_DIR.mkdir(parents=True, exist_ok=True)

PHASE_DIRS = {
    "phase0": OUTPUT_ROOT / "phase0",
    "phase1": OUTPUT_ROOT / "phase1",
    "phase2": OUTPUT_ROOT / "phase2",
    "phase3": OUTPUT_ROOT / "phase3",
    "phase4": OUTPUT_ROOT / "phase4",
    "phase5": OUTPUT_ROOT / "phase5_reliability_fusion",
}

COST_BPS_LIST = [0, 5, 10, 20, 50]
PRIMARY_COST_BPS = 10
TRADING_DAYS_PER_YEAR = 365
RUN_BLOCK_BOOTSTRAP = True
BOOTSTRAP_REPS = 2000
BOOTSTRAP_BLOCK_LENGTH = 7
BOOTSTRAP_SEED = 42


def load_all_ensemble_predictions() -> pd.DataFrame:
    frames = []
    for phase_name, directory in PHASE_DIRS.items():
        path = directory / "ensemble_predictions.csv"
        if not path.exists():
            print(f"Skipping {phase_name}: {path} not found")
            continue
        frame = pd.read_csv(path)
        frame["source_phase_directory"] = phase_name
        frames.append(frame)
    if not frames:
        raise FileNotFoundError("No ensemble_predictions.csv files were found in revised_outputs_v3.")

    combined = pd.concat(frames, ignore_index=True, sort=False)
    combined["target_date"] = pd.to_datetime(combined["target_date"], errors="coerce")
    combined = combined.dropna(subset=["target_date", "actual_return", "predicted_return"])

    # The same benchmark may appear in several phase files. Keep one copy per asset/method/date.
    combined = combined.sort_values(["asset", "method", "target_date", "source_phase_directory"])
    combined = combined.drop_duplicates(subset=["asset", "method", "target_date"], keep="last")
    return combined.reset_index(drop=True)


predictions = load_all_ensemble_predictions()
print("Methods available:")
for asset in sorted(predictions["asset"].unique()):
    methods = sorted(predictions.loc[predictions["asset"] == asset, "method"].unique())
    print(asset, methods)


In [ ]:
def choose_daily_sign_positions(predicted_log_returns: np.ndarray) -> np.ndarray:
    """Map predicted return sign directly to -1/0/+1 positions."""
    predicted = np.asarray(predicted_log_returns, dtype=float)
    return np.sign(predicted).astype(float)


def choose_cost_aware_positions(predicted_log_returns: np.ndarray, one_way_cost: float) -> np.ndarray:
    """Choose -1/0/+1 positions using predicted next-day return net of switching cost."""
    predicted_simple = np.expm1(np.asarray(predicted_log_returns, dtype=float))
    positions = np.zeros(len(predicted_simple), dtype=float)
    previous = 0.0
    candidates = np.array([-1.0, 0.0, 1.0])

    for i, forecast in enumerate(predicted_simple):
        utilities = candidates * forecast - one_way_cost * np.abs(candidates - previous)
        best_value = np.max(utilities)
        best_candidates = candidates[np.isclose(utilities, best_value)]
        if previous in best_candidates:
            chosen = previous
        elif 0.0 in best_candidates:
            chosen = 0.0
        else:
            chosen = best_candidates[0]
        positions[i] = chosen
        previous = chosen
    return positions


def calculate_max_drawdown(returns: np.ndarray) -> float:
    wealth = np.cumprod(1.0 + np.asarray(returns, dtype=float))
    if len(wealth) == 0:
        return np.nan
    peak = np.maximum.accumulate(wealth)
    drawdown = wealth / peak - 1.0
    return float(np.min(drawdown))


def summarize_returns(net_returns: np.ndarray, positions: np.ndarray, turnover: np.ndarray) -> dict:
    net_returns = np.asarray(net_returns, dtype=float)
    n = len(net_returns)
    if n == 0:
        return {}
    wealth = np.cumprod(1.0 + net_returns)
    total_return = float(wealth[-1] - 1.0)
    annualized_return = float(wealth[-1] ** (TRADING_DAYS_PER_YEAR / n) - 1.0) if wealth[-1] > 0 else np.nan
    annualized_volatility = float(np.std(net_returns, ddof=1) * np.sqrt(TRADING_DAYS_PER_YEAR)) if n > 1 else np.nan
    sharpe = (
        float(np.mean(net_returns) / np.std(net_returns, ddof=1) * np.sqrt(TRADING_DAYS_PER_YEAR))
        if n > 1 and np.std(net_returns, ddof=1) > 0 else np.nan
    )
    return {
        "observations": n,
        "total_return": total_return,
        "annualized_return": annualized_return,
        "annualized_volatility": annualized_volatility,
        "sharpe_ratio": sharpe,
        "maximum_drawdown": calculate_max_drawdown(net_returns),
        "mean_daily_net_return": float(np.mean(net_returns)),
        "positive_net_day_fraction": float(np.mean(net_returns > 0)),
        "average_absolute_position": float(np.mean(np.abs(positions))),
        "average_daily_turnover": float(np.mean(turnover)),
        "total_turnover": float(np.sum(turnover)),
        "position_changes": int(np.sum(turnover > 0)),
    }


def run_model_backtest(group: pd.DataFrame, cost_bps: float, policy: str) -> tuple[pd.DataFrame, dict]:
    data = group.sort_values("target_date").copy()
    actual_simple = np.expm1(data["actual_return"].to_numpy(dtype=float))
    predicted_log = data["predicted_return"].to_numpy(dtype=float)
    cost = cost_bps / 10000.0

    if policy == "daily_sign":
        positions = choose_daily_sign_positions(predicted_log)
    elif policy == "cost_aware":
        positions = choose_cost_aware_positions(predicted_log, one_way_cost=cost)
    else:
        raise ValueError(f"Unknown policy: {policy}")

    previous_positions = np.r_[0.0, positions[:-1]]
    turnover = np.abs(positions - previous_positions)
    trading_cost = cost * turnover
    gross_return = positions * actual_simple
    net_return = gross_return - trading_cost

    detail = data[["asset", "method", "target_date", "actual_return", "predicted_return"]].copy()
    detail["policy"] = policy
    detail["strategy"] = detail["method"].astype(str) + "__" + policy
    detail["actual_simple_return"] = actual_simple
    detail["position"] = positions
    detail["turnover"] = turnover
    detail["trading_cost"] = trading_cost
    detail["gross_strategy_return"] = gross_return
    detail["net_strategy_return"] = net_return
    detail["wealth"] = np.cumprod(1.0 + net_return)
    detail["cost_bps"] = cost_bps

    metrics = summarize_returns(net_return, positions, turnover)
    metrics.update({
        "asset": data["asset"].iloc[0],
        "method": data["method"].iloc[0],
        "policy": policy,
        "strategy": f"{data['method'].iloc[0]}__{policy}",
        "cost_bps": cost_bps,
    })
    return detail, metrics


def run_buy_hold_benchmark(asset_data: pd.DataFrame, cost_bps: float) -> tuple[pd.DataFrame, dict]:
    data = asset_data.sort_values("target_date").drop_duplicates("target_date").copy()
    actual_simple = np.expm1(data["actual_return"].to_numpy(dtype=float))
    cost = cost_bps / 10000.0
    positions = np.ones(len(data), dtype=float)
    turnover = np.zeros(len(data), dtype=float)
    if len(turnover):
        turnover[0] = 1.0
    net_return = actual_simple - cost * turnover

    detail = data[["asset", "target_date", "actual_return"]].copy()
    detail["method"] = "buy_and_hold"
    detail["policy"] = "benchmark"
    detail["strategy"] = "buy_and_hold"
    detail["predicted_return"] = np.nan
    detail["actual_simple_return"] = actual_simple
    detail["position"] = positions
    detail["turnover"] = turnover
    detail["trading_cost"] = cost * turnover
    detail["gross_strategy_return"] = actual_simple
    detail["net_strategy_return"] = net_return
    detail["wealth"] = np.cumprod(1.0 + net_return)
    detail["cost_bps"] = cost_bps

    metrics = summarize_returns(net_return, positions, turnover)
    metrics.update({"asset": data["asset"].iloc[0], "method": "buy_and_hold", "policy": "benchmark", "strategy": "buy_and_hold", "cost_bps": cost_bps})
    return detail, metrics


def run_cash_benchmark(asset_data: pd.DataFrame, cost_bps: float) -> tuple[pd.DataFrame, dict]:
    data = asset_data.sort_values("target_date").drop_duplicates("target_date").copy()
    positions = np.zeros(len(data), dtype=float)
    turnover = np.zeros(len(data), dtype=float)
    net_return = np.zeros(len(data), dtype=float)

    detail = data[["asset", "target_date", "actual_return"]].copy()
    detail["method"] = "cash"
    detail["policy"] = "benchmark"
    detail["strategy"] = "cash"
    detail["predicted_return"] = 0.0
    detail["actual_simple_return"] = np.expm1(data["actual_return"].to_numpy(dtype=float))
    detail["position"] = positions
    detail["turnover"] = turnover
    detail["trading_cost"] = 0.0
    detail["gross_strategy_return"] = 0.0
    detail["net_strategy_return"] = 0.0
    detail["wealth"] = 1.0
    detail["cost_bps"] = cost_bps

    metrics = summarize_returns(net_return, positions, turnover)
    metrics.update({"asset": data["asset"].iloc[0], "method": "cash", "policy": "benchmark", "strategy": "cash", "cost_bps": cost_bps})
    return detail, metrics


In [ ]:
all_details = []
all_metrics = []

for cost_bps in COST_BPS_LIST:
    for asset in sorted(predictions["asset"].unique()):
        asset_data = predictions[predictions["asset"] == asset].copy()
        for method, group in asset_data.groupby("method", sort=False):
            for policy in ["daily_sign", "cost_aware"]:
                detail, metrics = run_model_backtest(group, cost_bps, policy)
                all_details.append(detail)
                all_metrics.append(metrics)

        buy_hold_detail, buy_hold_metrics = run_buy_hold_benchmark(asset_data, cost_bps)
        cash_detail, cash_metrics = run_cash_benchmark(asset_data, cost_bps)
        all_details.extend([buy_hold_detail, cash_detail])
        all_metrics.extend([buy_hold_metrics, cash_metrics])

backtest_daily = pd.concat(all_details, ignore_index=True, sort=False)
backtest_metrics = pd.DataFrame(all_metrics)

backtest_daily.to_csv(BACKTEST_DIR / "backtest_daily.csv", index=False)
backtest_metrics.to_csv(BACKTEST_DIR / "backtest_metrics.csv", index=False)

primary = backtest_metrics[backtest_metrics["cost_bps"] == PRIMARY_COST_BPS].copy()
primary = primary.sort_values(["asset", "sharpe_ratio"], ascending=[True, False])
primary.to_csv(BACKTEST_DIR / f"backtest_primary_{PRIMARY_COST_BPS}bps.csv", index=False)

display(primary[[
    "asset", "strategy", "total_return", "annualized_return", "annualized_volatility",
    "sharpe_ratio", "maximum_drawdown", "average_daily_turnover", "position_changes"
]])


In [ ]:
def moving_block_bootstrap_mean_difference(
    candidate: np.ndarray,
    reference: np.ndarray,
    reps: int = 2000,
    block_length: int = 7,
    seed: int = 42,
) -> dict:
    """Moving-block bootstrap CI for candidate minus reference mean daily net return."""
    candidate = np.asarray(candidate, dtype=float)
    reference = np.asarray(reference, dtype=float)
    n = min(len(candidate), len(reference))
    candidate = candidate[:n]
    reference = reference[:n]
    diff = candidate - reference
    if n == 0:
        return {"mean_difference": np.nan, "ci_low": np.nan, "ci_high": np.nan}

    rng = np.random.default_rng(seed)
    starts = np.arange(max(n - block_length + 1, 1))
    boot_means = []
    for _ in range(reps):
        sampled = []
        while len(sampled) < n:
            start = int(rng.choice(starts))
            block = diff[start:start + block_length]
            sampled.extend(block.tolist())
        boot_means.append(float(np.mean(sampled[:n])))

    return {
        "mean_difference": float(np.mean(diff)),
        "ci_low": float(np.quantile(boot_means, 0.025)),
        "ci_high": float(np.quantile(boot_means, 0.975)),
    }


bootstrap_rows = []
if RUN_BLOCK_BOOTSTRAP:
    primary_daily = backtest_daily[backtest_daily["cost_bps"] == PRIMARY_COST_BPS].copy()
    for asset in sorted(primary_daily["asset"].unique()):
        asset_daily = primary_daily[primary_daily["asset"] == asset]
        cash = asset_daily[asset_daily["strategy"] == "cash"].sort_values("target_date")
        strategies = sorted(s for s in asset_daily["strategy"].unique() if s.endswith("__daily_sign"))
        for strategy in strategies:
            candidate = asset_daily[asset_daily["strategy"] == strategy].sort_values("target_date")
            paired = candidate[["target_date", "net_strategy_return"]].merge(
                cash[["target_date", "net_strategy_return"]],
                on="target_date",
                suffixes=("_candidate", "_cash"),
            )
            stats = moving_block_bootstrap_mean_difference(
                paired["net_strategy_return_candidate"].to_numpy(),
                paired["net_strategy_return_cash"].to_numpy(),
                reps=BOOTSTRAP_REPS,
                block_length=BOOTSTRAP_BLOCK_LENGTH,
                seed=BOOTSTRAP_SEED,
            )
            bootstrap_rows.append({
                "asset": asset,
                "strategy": strategy,
                "reference": "cash",
                "cost_bps": PRIMARY_COST_BPS,
                **stats,
            })

bootstrap_table = pd.DataFrame(bootstrap_rows)
bootstrap_table.to_csv(BACKTEST_DIR / "block_bootstrap_mean_return_ci.csv", index=False)
display(bootstrap_table)


In [ ]:
# Cumulative wealth plots at the primary cost assumption.
# Use the daily-sign strategy as the primary economic sanity check.
REPRESENTATIVE_METHODS = [
    "phase0_lstm",
    "phase2_general_lstm",
    "phase3_crypto_lstm",
    "phase4_decay_only_lstm",
    "phase4_combined_lstm",
    "phase5_reliability_fusion_lstm",
]

primary_daily = backtest_daily[backtest_daily["cost_bps"] == PRIMARY_COST_BPS].copy()
for asset in sorted(primary_daily["asset"].unique()):
    asset_data = primary_daily[primary_daily["asset"] == asset]
    fig, ax = plt.subplots(figsize=(13, 5))
    for method in REPRESENTATIVE_METHODS:
        strategy = f"{method}__daily_sign"
        method_data = asset_data[asset_data["strategy"] == strategy].sort_values("target_date")
        if not method_data.empty:
            ax.plot(method_data["target_date"], method_data["wealth"], label=strategy, linewidth=1.1)
    for benchmark in ["buy_and_hold", "cash"]:
        method_data = asset_data[asset_data["strategy"] == benchmark].sort_values("target_date")
        if not method_data.empty:
            ax.plot(method_data["target_date"], method_data["wealth"], label=benchmark, linewidth=1.1)
    ax.set_title(f"{asset}: Daily-Sign Strategy after {PRIMARY_COST_BPS} bps One-Way Trading Costs")
    ax.set_xlabel("Date")
    ax.set_ylabel("Wealth (initial = 1.0)")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(BACKTEST_DIR / f"cumulative_wealth_daily_sign_{asset}_{PRIMARY_COST_BPS}bps.png", dpi=300, bbox_inches="tight")
    fig.savefig(BACKTEST_DIR / f"cumulative_wealth_daily_sign_{asset}_{PRIMARY_COST_BPS}bps.pdf", bbox_inches="tight")
    plt.show()

# Cost-sensitivity chart for annualized return using daily-sign policy.
for asset in sorted(backtest_metrics["asset"].unique()):
    asset_metrics = backtest_metrics[(backtest_metrics["asset"] == asset) & (backtest_metrics["policy"] == "daily_sign")].copy()
    fig, ax = plt.subplots(figsize=(11, 5))
    for method in REPRESENTATIVE_METHODS:
        method_data = asset_metrics[asset_metrics["method"] == method].sort_values("cost_bps")
        if not method_data.empty:
            ax.plot(method_data["cost_bps"], method_data["annualized_return"], marker="o", label=method)
    ax.axhline(0.0, linestyle="--", linewidth=0.8)
    ax.set_title(f"{asset}: Daily-Sign Annualized Return Sensitivity to Trading Costs")
    ax.set_xlabel("One-way trading cost (bps)")
    ax.set_ylabel("Annualized return")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(BACKTEST_DIR / f"cost_sensitivity_daily_sign_{asset}.png", dpi=300, bbox_inches="tight")
    fig.savefig(BACKTEST_DIR / f"cost_sensitivity_daily_sign_{asset}.pdf", bbox_inches="tight")
    plt.show()

print(f"Backtest outputs saved to: {BACKTEST_DIR}")


## Interpretation rules for the paper

Use the **daily-sign strategy** as the primary economic sanity check because it is the direct trading interpretation of Directional Accuracy and contains no tuned threshold. Treat the **cost-aware strategy** as a secondary robustness check.

A statistically lower MAE does **not** automatically imply economic value. For an economically meaningful result, look for a method that:

- remains profitable after nonzero trading costs,
- has a positive or competitive Sharpe ratio,
- does not require extreme turnover,
- does not rely on one cost assumption,
- and preferably shows the same qualitative result under later rolling/backtest extensions.

If the daily-sign strategy becomes unprofitable after 5-20 bps of one-way costs, the correct conclusion is that the forecast improvement is statistically detectable but economically weak. That is still a useful and credible result.